# Part A — Data Foundation: Clean, Store, Query

This notebook solves all seven Part A tasks using the shared, tested Python modules. All records, policies and brand context are synthetic. Amounts are in **INR**. Run cells from top to bottom; no charts or image output are produced. The notebook writes clean CSV files, the SQLite database, seven SQL result CSVs and an actual-results JSON report.

**Inputs:** `data/raw/customers.csv`, `products.csv`, `orders.csv`. **Outputs:** `data/clean/`, `data/instamart.db`, `outputs/sql_q1.csv` through `sql_q7.csv`, `outputs/part_a_report.json`.

## 1. Reproducible environment and supplied data

Create an isolated environment and install `requirements.txt` using the README commands. This cell discovers the repository from either the root or the Part A folder, checks Python/SQLite versions, and generates the supplied data only if it is missing. The supplied generator and mock LLM remain unmodified.

In [1]:
from pathlib import Path
import os, sys, subprocess, sqlite3, hashlib, json

working = Path.cwd().resolve()
ROOT = next((p for p in [working, *working.parents]
             if (p / 'generate_data.py').is_file() and (p / 'part_a').is_dir()), None)
assert ROOT is not None, 'Open this notebook inside the project repository.'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info >= (3, 11)
assert sqlite3.sqlite_version_info >= (3, 25, 0)
if not Path('data/raw/orders.csv').exists():
    subprocess.run([sys.executable, 'generate_data.py'], check=True)
import pandas as pd
import numpy as np
print('Python:', sys.version.split()[0], 'SQLite:', sqlite3.sqlite_version)
print('pandas:', pd.__version__, 'numpy:', np.__version__)
for filename in ['generate_data.py', 'mock_llm.py']:
    print(filename, 'SHA256:', hashlib.sha256(Path(filename).read_bytes()).hexdigest())

Python: 3.12.13 SQLite: 3.50.4
pandas: 3.1.0 numpy: 2.5.4
generate_data.py SHA256: e5bd939dfa40402281d0285bdf755d3b75f42159265d78098e248e58683de6cd
mock_llm.py SHA256: 77bf9bb7cb96e67f49d808cb2e84befb0e5a346833cd508df924437b606905b6


## 2. Audit every raw column

Report table shapes, data types, missing counts and percentages. `N/A` may be parsed as missing by pandas; the separate nonnumeric-amount checkpoint also counts the string `error`. Tickets use `keep_default_na=False` so empty strings remain empty strings in Part C.

In [2]:
from part_a.audit import audit_raw_data
shapes, audit, column_audits = audit_raw_data()
print('TABLE SHAPES:', shapes)
for table in ['customers', 'products', 'orders']:
    print('\nCOLUMN AUDIT:', table)
    print(column_audits[table].to_string(index=False))
print('\nRAW CHECKPOINTS:')
print(pd.Series(audit).to_string())
expected_audit = {
    'raw_order_rows':13084, 'exact_duplicates':119, 'pending_rows':656,
    'distinct_order_ids':12309, 'non_numeric_amounts':120,
    'negative_delivery_times':106, 'blank_delivery_times':2541,
    'blank_ages':61, 'ages_outside_range':6, 'distinct_raw_cities':24,
    'distinct_raw_genders':10, 'blank_genders':30,
}
assert audit == expected_audit
print('All raw audit references matched.')

TABLE SHAPES: {'customers_shape': (1200, 5), 'products_shape': (40, 4), 'orders_shape': (13084, 9), 'tickets_shape': (164, 5)}

COLUMN AUDIT: customers
     column   dtype  missing_count  missing_pct
customer_id     str              0         0.00
       city     str              0         0.00
signup_date     str              0         0.00
        age float64             61         5.08
     gender     str             30         2.50

COLUMN AUDIT: products
      column dtype  missing_count  missing_pct
  product_id   str              0          0.0
product_name   str              0          0.0
    category   str              0          0.0
   price_inr int64              0          0.0

COLUMN AUDIT: orders
          column   dtype  missing_count  missing_pct
        order_id     str              0         0.00
     customer_id     str              0         0.00
      product_id     str              0         0.00
        quantity   int64              0         0.00
        order_

## 3. Clean customer dimensions

Strip and title-case cities; map gender spellings to `male`, `female`, `unknown`; replace ages outside 10–100 with missing; fill missing ages with city medians using `groupby(...).transform('median')`; parse signup dates. Preserve fractional medians rather than rounding them to an integer.

In [3]:
from part_a.clean import clean_customers_data, clean_orders_data
from part_a.report import cleaning_changes
raw_customers = pd.read_csv('data/raw/customers.csv')
raw_products = pd.read_csv('data/raw/products.csv')
raw_orders = pd.read_csv('data/raw/orders.csv')
customers = clean_customers_data(raw_customers)
products = raw_products.copy()
print('City counts:', customers.city.value_counts().to_dict())
print('Gender counts:', customers.gender.value_counts().to_dict())
print('Missing ages:', int(customers.age.isna().sum()))
print('Age range:', float(customers.age.min()), float(customers.age.max()))
print('Signup dtype:', customers.signup_date.dtype)
changes = cleaning_changes()
print('Values changed:', {k:v for k,v in changes.items() if 'city' in k or 'gender' in k or 'age' in k})

City counts: {'Pune': 227, 'Delhi': 210, 'Bengaluru': 197, 'Chennai': 191, 'Hyderabad': 191, 'Mumbai': 184}
Gender counts: {'female': 610, 'male': 560, 'unknown': 30}
Missing ages: 0
Age range: 18.0 60.0
Signup dtype: datetime64[us]
Values changed: {'city_values_standardized': 476, 'gender_values_standardized': 950, 'blank_gender_to_unknown': 30, 'invalid_ages_to_missing': 6, 'ages_imputed_city_median': 67, 'city_age_medians': {'Bengaluru': 42.0, 'Chennai': 39.0, 'Delhi': 43.0, 'Hyderabad': 37.5, 'Mumbai': 38.5, 'Pune': 39.0}}


## 4. Clean order records in the required order

1. Parse timestamps, then remove exact duplicates.
2. Keep the complete latest row for each order id, including its missing values.
3. Parse amounts and reconstruct missing values as `round(price × quantity × (1 − discount / 100), 2)`.
4. Replace negative delivery durations with missing and impute completed orders using their completed-category median.
5. Keep cancelled delivery durations missing.

Raw counts differ from change counts because duplicates are removed before imputing. Products need no changes.

In [4]:
orders = clean_orders_data(raw_orders, products)
completed = orders[orders.status.eq('completed')]
cancelled = orders[orders.status.eq('cancelled')]
print('Clean orders:', len(orders), 'distinct IDs:', orders.order_id.nunique())
print('Final statuses:', orders.status.value_counts().to_dict())
print('Missing amounts:', int(orders.order_amount.isna().sum()))
print('Missing completed delivery:', int(completed.delivery_minutes.isna().sum()))
print('Missing cancelled delivery:', int(cancelled.delivery_minutes.isna().sum()))
print(f'Completed revenue: INR {completed.order_amount.sum():,.2f}')
print('Exact-stage cleaning counts:')
print(json.dumps(changes, indent=2))

Clean orders: 12309 distinct IDs: 12309
Final statuses: {'completed': 11110, 'cancelled': 1199}
Missing amounts: 0
Missing completed delivery: 0
Missing cancelled delivery: 1199
Completed revenue: INR 2,998,668.80
Exact-stage cleaning counts:
{
  "city_values_standardized": 476,
  "gender_values_standardized": 950,
  "blank_gender_to_unknown": 30,
  "invalid_ages_to_missing": 6,
  "ages_imputed_city_median": 67,
  "signup_dates_parsed": 1200,
  "order_timestamps_parsed": 13084,
  "exact_duplicate_rows_removed": 119,
  "older_order_records_removed": 656,
  "amounts_reconstructed": 112,
  "negative_delivery_to_missing": 105,
  "completed_delivery_imputed": 774,
  "cancelled_delivery_left_missing": 1199,
  "products_changed": 0,
  "category_delivery_medians": {
    "Dairy": 25.0,
    "Electronics": 25.0,
    "Grocery": 25.0,
    "Personal Care": 25.5,
    "Snacks": 25.0
  },
  "city_age_medians": {
    "Bengaluru": 42.0,
    "Chennai": 39.0,
    "Delhi": 43.0,
    "Hyderabad": 37.5,
    "

## 5. Persist and validate the analytical foundation

Write date strings in the required format, store exactly the three tables in SQLite, and run more than eight reference assertions. The standalone validator prints `ALL CHECKPOINTS PASSED`.

In [5]:
from part_a.db import store_to_sqlite, run_sql_queries
from part_a.validate import validate_part_a
Path('data/clean').mkdir(parents=True, exist_ok=True)
customers.to_csv('data/clean/customers.csv', index=False, date_format='%Y-%m-%d')
products.to_csv('data/clean/products.csv', index=False)
orders.to_csv('data/clean/orders.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')
store_to_sqlite(customers, products, orders)
validate_part_a()
with sqlite3.connect('data/instamart.db') as connection:
    print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection).to_string(index=False))
    print(pd.read_sql_query('SELECT order_ts, status, delivery_minutes FROM orders LIMIT 3', connection).to_string(index=False))

ALL CHECKPOINTS PASSED
     name
customers
   orders
 products
           order_ts    status  delivery_minutes
2025-09-16 08:07:21 completed              20.0
2025-09-03 07:34:01 completed              23.0
2025-04-08 17:08:14 completed              23.0


## 6. Seven business questions in SQL

The full SQL lives in `sql/queries.sql`. Only completed orders count as revenue or spend. Each following cell reports its query and checks the required outcome.

In [6]:
from part_a.report import validate_sql_results
sql_results = run_sql_queries()
assert validate_sql_results(sql_results)
print('All seven SQL reference checkpoints passed.')
print(Path('sql/queries.sql').read_text())

All seven SQL reference checkpoints passed.
-- ============================================================================
-- Instamart Customer Intelligence & Support Copilot - SQL Queries
-- ============================================================================

-- Q1: WHERE then HAVING
-- Keep only completed orders with order_amount > 500 (WHERE),
-- group by customer, and keep customers whose total of those orders exceeds 8000 (HAVING).
-- Reference: 30 customers.
SELECT
    customer_id,
    ROUND(SUM(order_amount), 2) AS total_high_value_spend
FROM orders
WHERE status = 'completed' AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000;

-- Q2: Three-Table Join
-- Revenue by city and product category (join orders, customers, products).
-- Reference: 30 rows, the 30 revenues add up to INR 2,998,668.80,
-- the largest cell is Bengaluru / Electronics at INR 233,909.45.
SELECT
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) AS revenue
FROM ord

### Q1 — Filter orders with WHERE, then customers with HAVING

Completed orders above INR 500 enter the aggregation; only customers with more than INR 8,000 of that filtered spend qualify.

In [7]:
print('Q1 qualifying customers:', len(sql_results['Q1']))
print(sql_results['Q1'].head(10).to_string(index=False))

Q1 qualifying customers: 30
customer_id  total_high_value_spend
      C0082                12317.70
      C0130                 9170.40
      C0147                 9141.50
      C0167                 8718.55
      C0267                12150.25
      C0291                12150.70
      C0308                 9872.40
      C0332                10779.50
      C0407                 9451.75
      C0424                 8294.20


### Q2 — Revenue by city and product category, joining all three tables

In [8]:
q2 = sql_results['Q2']
print(q2.to_string(index=False))
print(f'Total across {len(q2)} cells: INR {q2.revenue.sum():,.2f}')
print('Largest cell:', q2.loc[q2.revenue.idxmax()].to_dict())

     city      category   revenue
Bengaluru         Dairy  87075.05
Bengaluru   Electronics 233909.45
Bengaluru       Grocery 110915.65
Bengaluru Personal Care  20193.85
Bengaluru        Snacks  71549.40
  Chennai         Dairy  78301.80
  Chennai   Electronics 195508.80
  Chennai       Grocery  93817.45
  Chennai Personal Care  27675.40
  Chennai        Snacks  68325.70
    Delhi         Dairy 117271.70
    Delhi   Electronics 158549.10
    Delhi       Grocery 112658.05
    Delhi Personal Care  23471.50
    Delhi        Snacks  90879.45
Hyderabad         Dairy  83731.00
Hyderabad   Electronics 228913.10
Hyderabad       Grocery 109475.40
Hyderabad Personal Care  24800.05
Hyderabad        Snacks  81670.35
   Mumbai         Dairy  75209.85
   Mumbai   Electronics 214738.40
   Mumbai       Grocery  78970.15
   Mumbai Personal Care  38018.00
   Mumbai        Snacks  72122.10
     Pune         Dairy  90349.55
     Pune   Electronics 187629.25
     Pune       Grocery 108456.60
     Pune Pers

### Q3 — Catalogue prices above their scalar-subquery average

In [9]:
print('Q3 products:', len(sql_results['Q3']))
print(sql_results['Q3'].to_string(index=False))

Q3 products: 11
product_id        product_name      category  price_inr
      P014          Ghee 500ml         Dairy        340
      P030   Body Lotion 400ml Personal Care        350
      P032       Sunscreen 50g Personal Care        399
      P033    Wireless Earbuds   Electronics       1499
      P034 Power Bank 10000mAh   Electronics       1299
      P035      USB-C Cable 1m   Electronics        299
      P036   Phone Charger 20W   Electronics        799
      P037          Smart Bulb   Electronics        599
      P038   Bluetooth Speaker   Electronics       1999
      P039         Phone Stand   Electronics        349
      P040    Memory Card 64GB   Electronics        549


### Q4 — LEFT JOIN identifies customers with no completed order

In [10]:
print('Q4 customers with no completed order:', len(sql_results['Q4']))
print(sql_results['Q4'].to_string(index=False))

Q4 customers with no completed order: 25
customer_id      city signup_date
      C0029 Hyderabad  2024-03-21
      C0037      Pune  2024-01-23
      C0093      Pune  2025-03-16
      C0115 Bengaluru  2024-04-11
      C0120 Bengaluru  2024-03-08
      C0151      Pune  2024-09-13
      C0354   Chennai  2024-01-13
      C0363   Chennai  2024-10-11
      C0412     Delhi  2024-01-05
      C0461 Bengaluru  2024-12-29
      C0477    Mumbai  2024-04-17
      C0500 Hyderabad  2024-05-14
      C0607      Pune  2024-12-03
      C0625     Delhi  2024-01-26
      C0654 Hyderabad  2024-12-17
      C0724   Chennai  2024-08-20
      C0759     Delhi  2024-01-29
      C0827   Chennai  2025-01-12
      C0835 Hyderabad  2024-05-16
      C1041     Delhi  2024-12-30
      C1052     Delhi  2024-03-04
      C1083      Pune  2024-09-14
      C1111      Pune  2024-03-07
      C1165 Bengaluru  2024-08-14
      C1189 Bengaluru  2024-06-28


### Q5 — Two CTEs find customers above average completed spend

In [11]:
average_spend = completed.groupby('customer_id').order_amount.sum().mean()
print('Q5 qualifying customers:', len(sql_results['Q5']))
print(f'Average completed spend per active customer: INR {average_spend:,.2f}')
print(sql_results['Q5'].head(10).to_string(index=False))

Q5 qualifying customers: 550
Average completed spend per active customer: INR 2,552.06
customer_id  total_spend
      C0002      4644.90
      C0003      3644.85
      C0007      2600.00
      C0008      3096.50
      C0009      8424.80
      C0011      3583.85
      C0012      3395.30
      C0014      8006.90
      C0016      3546.00
      C0017      7521.70


### Q6 — RANK, DENSE_RANK and ROW_NUMBER within each city

Tied spending totals share a rank, while `ROW_NUMBER` still assigns unique positions; this produces eight rows where rank differs from row number.

In [12]:
q6 = sql_results['Q6']
print('City leaders:')
print(q6[q6.row_num.eq(1)].sort_values('city').to_string(index=False))
print('Rows with RANK different from ROW_NUMBER:', int(q6.rank_num.ne(q6.row_num).sum()))
print(q6[q6.rank_num.ne(q6.row_num)].to_string(index=False))

City leaders:
customer_id      city  total_spend  rank_num  dense_rank_num  row_num
      C0082 Bengaluru     12856.20         1               1        1
      C0930   Chennai     10151.40         1               1        1
      C0291     Delhi     13742.60         1               1        1
      C0743 Hyderabad     12730.05         1               1        1
      C0693    Mumbai     14766.55         1               1        1
      C0426      Pune     10377.10         1               1        1
Rows with RANK different from ROW_NUMBER: 8
customer_id      city  total_spend  rank_num  dense_rank_num  row_num
      C0734 Bengaluru      2598.00        99              99      100
      C0723 Bengaluru       135.00       179             178      180
      C1053    Mumbai       396.00       149             149      150
      C0608    Mumbai       280.00       160             159      161
      C1064      Pune      3236.95        74              74       75
      C0057      Pune       285.

### Q7 — Daily revenue and cumulative revenue with SUM OVER

In [13]:
q7 = sql_results['Q7']
print('Daily rows:', len(q7))
print(q7.head(5).to_string(index=False))
print(q7.tail(5).to_string(index=False))
print(f'Final cumulative revenue: INR {q7.cumulative_revenue.iloc[-1]:,.2f}')

Daily rows: 183
       day  revenue  cumulative_revenue
2025-04-01 16876.50            16876.50
2025-04-02 19854.55            36731.05
2025-04-03 28048.35            64779.40
2025-04-04 14249.60            79029.00
2025-04-05 16031.50            95060.50
       day  revenue  cumulative_revenue
2025-09-26 17190.85          2942763.75
2025-09-27 13689.95          2956453.70
2025-09-28 12689.85          2969143.55
2025-09-29 14022.15          2983165.70
2025-09-30 15503.10          2998668.80
Final cumulative revenue: INR 2,998,668.80


## 7. pandas cross-check and exploratory analysis

Rebuild Q2 with a pivot table and margins, use named city aggregations, calculate quartiles and IQR fences, and compare the four requested numeric features in a correlation matrix.

In [14]:
from part_a.eda import run_eda
eda = run_eda()
print('City/category pivot with margins:')
print(eda['pivot_table'].to_string())
assert abs(eda['grand_total'] - q2.revenue.sum()) < 0.01
print('\nCity sum/mean/count:')
print(eda['city_aggregation'].to_string())
print('\nDistribution statistics:')
print(pd.Series(eda['distribution_stats']).to_string())
print('\nCorrelation matrix:')
print(eda['correlation_matrix'].to_string())
from part_a.report import build_report
report = build_report()
print('Largest absolute correlation:', report['largest_absolute_correlation'])
print('Measured report written: outputs/part_a_report.json')

City/category pivot with margins:
category       Dairy  Electronics    Grocery  Personal Care     Snacks       Total
city                                                                              
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   523643.40
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   463629.15
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   502829.80
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   528589.90
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   479058.50
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   500918.05
Total      531938.95   1219248.10  614293.30      168433.95  464754.50  2998668.80

City sum/mean/count:
                 sum    mean  count
city                               
Bengaluru  523643.40  276.47   1894
Chennai    463629.15  272.72   1700
Delhi      502829.80  243.62   2064
Hyderabad  528589

### Business interpretation (under 200 words)

The data describes synthetic grocery and small-electronics transactions for 1,200 customers across six cities during April–September 2025. Completed revenue is INR 2,998,668.80 across 11,110 orders. The order-amount mean is INR 269.91 versus a median of INR 135.00, and skewness is 4.61, so a small number of expensive baskets stretch the right tail.

The first and third quartiles are INR 64.80 and INR 297.00. The IQR fences are INR −283.50 and INR 645.30; 897 orders exceed the upper fence and none fall below the lower fence. These high baskets need review in context, especially electronics, rather than automatic deletion. The planted impossible ages, negative delivery durations and malformed amounts are handled separately by documented rules.

Of the requested variables, amount and discount have the largest absolute correlation, approximately −0.095; amount and quantity are about +0.087. Both relationships are weak. A correlation of this size does not establish causation. City/category revenue and customer spending support segmentation in Part B; the same clean database supports refund lookups in Part D.